In [10]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from concurrent.futures import ThreadPoolExecutor, as_completed
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import subprocess
import os
import random
import json
import pickle

In [11]:
# ==================== SET FULL DETERMINISM ====================
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
os.environ['PYTHONHASHSEED'] = str(seed_value)

In [12]:
# ==================== KONFIGURASI ====================
ticker = "AAPL"
lookback = 5
epsilon = 1e-6
max_tweets_per_day = 100
covid_periods = {
    'Sebelum COVID': ('2017-04-09', '2020-02-29'),
    'Selama COVID':   ('2020-03-01', '2022-12-31'),
    'Setelah COVID':  ('2023-01-01', '2025-09-15')
}

base_path = "saved_models_arima"
os.makedirs(base_path, exist_ok=True)

sentiment_folder = "saved_sentiment_scraping"
os.makedirs(sentiment_folder, exist_ok=True)

# Path penyimpanan
raw_data_path = os.path.join(base_path, f"raw_stock_prices_{ticker.lower()}.csv")
sentiment_path = os.path.join(sentiment_folder, f"sentiment_{ticker.lower()}.csv")
processed_features_path = os.path.join(base_path, f"processed_features_{ticker.lower()}.csv")
determinism_log_path = os.path.join(base_path, "determinism_log.txt")

# Simpan log determinisme
with open(determinism_log_path, 'w') as f:
    f.write(f"Seed value: {seed_value}\n")
    f.write(f"PYTHONHASHSEED: {os.environ['PYTHONHASHSEED']}\n")
print(f"💾 Log determinisme disimpan di {determinism_log_path}")

💾 Log determinisme disimpan di saved_models_arima\determinism_log.txt


In [13]:
# ==================== FUNGSI MAPE ====================
def mape(y_true, y_pred):
    y = np.maximum(np.abs(y_true), epsilon)
    return np.mean(np.abs((y_true - y_pred) / y)) * 100

In [14]:
# ==================== DATA SAHAM ====================
if os.path.exists(raw_data_path):
    data = pd.read_csv(
        raw_data_path,
        skiprows=3, header=None,
        names=['Date','Close'],
        parse_dates=['Date']
    ).set_index('Date')
    data['Close'] = data['Close'].astype(float)
else:
    data = yf.download(ticker, start='2017-04-09', end='2025-09-16')[['Close']]
    data.to_csv(raw_data_path)

data_daily = data.resample('D').ffill().interpolate()

In [15]:
# ==================== SENTIMEN HARIAN ====================
if os.path.exists(sentiment_path):
    sdf = pd.read_csv(sentiment_path, parse_dates=['Date'], index_col='Date')
    print(f"ℹ️ Memuat sentimen dari {sentiment_path}")
else:
    print("⚠️ File sentimen tidak ditemukan, menjalankan Scraping Sentiment.ipynb ...")
    try:
        subprocess.run(
            ["jupyter", "nbconvert", "--to", "notebook", "--execute",
             "Scraping Sentiment.ipynb", "--output", "Scraping_Sentiment_Out.ipynb"],
            check=True
        )
        sdf = pd.read_csv(sentiment_path, parse_dates=['Date'], index_col='Date')
        print(f"💾 Sentimen berhasil dibuat di {sentiment_path}")
    except Exception as e:
        raise RuntimeError(f"Gagal menjalankan Scraping Sentiment.ipynb: {e}")

# Forward-fill untuk sentimen agar sejajar dengan data saham
data_daily['Sentiment'] = sdf['Sentiment'].reindex(data_daily.index).fillna(method='ffill')
data_daily['Sentiment_Lag1'] = data_daily['Sentiment'].shift(1).fillna(method='ffill')


ℹ️ Memuat sentimen dari saved_sentiment_scraping\sentiment_aapl.csv


C:\Users\andil\AppData\Local\Temp\ipykernel_15796\2983231727.py:19: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_daily['Sentiment'] = sdf['Sentiment'].reindex(data_daily.index).fillna(method='ffill')
C:\Users\andil\AppData\Local\Temp\ipykernel_15796\2983231727.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_daily['Sentiment_Lag1'] = data_daily['Sentiment'].shift(1).fillna(method='ffill')


In [16]:
# ==================== FITUR TEKNIKAL ====================
if os.path.exists(processed_features_path):
    data_daily = pd.read_csv(processed_features_path, parse_dates=['Date'], index_col='Date')
    print(f"ℹ️ Memuat fitur terproses dari {processed_features_path}")
else:
    for i in range(1, lookback+1):
        data_daily[f'lag_{i}'] = data_daily['Close'].shift(i)
    data_daily['MA_5'] = data_daily['Close'].rolling(5).mean()
    data_daily['MA_20'] = data_daily['Close'].rolling(20).mean()
    data_daily['Return_1D'] = data_daily['Close'].pct_change()
    data_daily['Volatility_5D'] = data_daily['Return_1D'].rolling(5).std()

    def compute_rsi(s, period=14):
        delta = s.diff()
        gain = delta.where(delta>0, 0).rolling(period).mean()
        loss = (-delta.where(delta<0, 0)).rolling(period).mean()
        rs = gain / (loss + epsilon)
        return 100 - (100/(1+rs))

    data_daily['RSI_14'] = compute_rsi(data_daily['Close'])
    data_daily['Momentum_5D'] = data_daily['Close'] - data_daily['Close'].shift(5)
    data_daily['MA_5_20_ratio'] = data_daily['MA_5'] / (data_daily['MA_20'] + epsilon)
    data_daily['RSI_MA5'] = data_daily['RSI_14'] / data_daily['MA_5']

    data_daily.dropna(inplace=True)
    data_daily.to_csv(processed_features_path)
    print(f"💾 Data fitur proses disimpan di {processed_features_path}")

ℹ️ Memuat fitur terproses dari saved_models_arima\processed_features_aapl.csv


In [17]:
# ==================== EXOGENOUS FEATURES ====================
exog_features = [
    'Sentiment_Lag1','MA_5','MA_20','Return_1D',
    'Volatility_5D','RSI_14','Momentum_5D',
    'MA_5_20_ratio','RSI_MA5'
]

# -------------------
# Helper: model & scaler paths per period
# -------------------
def model_path_for(period_name):
    return os.path.join(base_path, f"model_arima_{period_name.replace(' ','_').lower()}.pkl")

def scaler_path_for(period_name):
    return os.path.join(base_path, f"scaler_exog_{period_name.replace(' ','_').lower()}.pkl")

def params_path_for(period_name):
    return os.path.join(base_path, f"params_arima_{ticker.lower()}_{period_name.lower().replace(' ','_')}.json")

In [18]:
# ==================== ARIMA PER PERIODE (Load/Fit Scaler & Model) ====================
def process_arima_period(period_name, start_date, end_date):
    param_path = params_path_for(period_name)
    model_path = model_path_for(period_name)
    scaler_path = scaler_path_for(period_name)
    plot_path = os.path.join(base_path, f"plot_{period_name.replace(' ','_').lower()}.png")

    dfp = data_daily.loc[start_date:end_date].copy().dropna()
    split = int(len(dfp) * 0.8)
    train = dfp.iloc[:split]
    test = dfp.iloc[split:]

    # ----------------- Load atau Fit Scaler -----------------
    if os.path.exists(scaler_path):
        with open(scaler_path, 'rb') as f:
            scaler_exog = pickle.load(f)
        train_exog_scaled = scaler_exog.transform(train[exog_features])
        test_exog_scaled = scaler_exog.transform(test[exog_features])
        print(f"ℹ️ Memuat scaler dari {scaler_path}")
    else:
        scaler_exog = MinMaxScaler()
        train_exog_scaled = scaler_exog.fit_transform(train[exog_features])
        test_exog_scaled = scaler_exog.transform(test[exog_features])
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler_exog, f)
        print(f"💾 Scaler disimpan di {scaler_path}")

    # ----------------- Load atau Fit Model -----------------
    if os.path.exists(param_path):
        order = tuple(json.load(open(param_path))['order'])
    else:
        am = auto_arima(
            train['Close'],
            exogenous=train_exog_scaled,
            seasonal=False,
            stepwise=True,
            random_state=seed_value
        )
        order = am.order
        json.dump({'order': order}, open(param_path, 'w'))

    if os.path.exists(model_path):
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print(f"ℹ️ Memuat model dari {model_path}")
    else:
        model = ARIMA(train['Close'], order=order, exog=train_exog_scaled).fit()
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print(f"💾 Model ARIMA disimpan di {model_path}")


    # Rolling forecast
    hist = train['Close'].values.tolist()
    exog_hist = train_exog_scaled.tolist()
    cur_mod = model
    preds = []
    
    for i in range(len(test)):
        xf = test_exog_scaled[i].reshape(1, -1)
        f = cur_mod.forecast(steps=1, exog=xf)
        preds.append(f[0])
        hist.append(test['Close'].iloc[i])
        exog_hist.append(xf.flatten().tolist())
        if (i + 1) % 30 == 0:
            cur_mod = ARIMA(hist, order=order, exog=exog_hist).fit()


    plt.figure(figsize=(10, 5))
    plt.plot(test.index, test['Close'].values, label='Aktual')
    plt.plot(test.index, preds, '--', label='Prediksi')
    plt.title(f'ARIMA {period_name}')
    plt.legend()
    plt.grid(True)
    plt.savefig(plot_path, dpi=300)
    plt.close()

    return {
        'dates': test.index,
        'actual': test['Close'].values,
        'pred': np.array(preds)
    }

if __name__ == "__main__":
    results, metrics = {}, {}

    for p, (s, e) in covid_periods.items():
        res = process_arima_period(p, s, e)
        results[p] = res
        y_t, y_p = res['actual'], res['pred']
        metrics[p] = {
            'MSE': mean_squared_error(y_t, y_p),
            'MAE': mean_absolute_error(y_t, y_p),
            'R2': r2_score(y_t, y_p),
            'MAPE': mape(y_t, y_p)
        }

    # ==================== MODEL KESELURUHAN ====================
    param_all = os.path.join(base_path, f"params_arima_{ticker.lower()}_keseluruhan.json")
    scaler_all_path = os.path.join(base_path, 'scaler_exog_keseluruhan.pkl')
    model_all_path = os.path.join(base_path, "model_arima_keseluruhan.pkl")

    # ----------------- Load atau Fit Scaler -----------------
    if os.path.exists(scaler_all_path):
        with open(scaler_all_path, 'rb') as f:
            scaler_all = pickle.load(f)
        exog_all_scaled = scaler_all.transform(data_daily[exog_features])
        print(f"ℹ️ Memuat scaler keseluruhan dari {scaler_all_path}")
    else:
        scaler_all = MinMaxScaler()
        exog_all_scaled = scaler_all.fit_transform(data_daily[exog_features])
        with open(scaler_all_path, 'wb') as f:
            pickle.dump(scaler_all, f)
        print(f"💾 Scaler keseluruhan disimpan di {scaler_all_path}")

    # ----------------- Load atau Fit Model -----------------
    if os.path.exists(model_all_path):
        with open(model_all_path, 'rb') as f:
            model_all = pickle.load(f)
        print(f"ℹ️ Memuat model keseluruhan dari {model_all_path}")
    else:
        if os.path.exists(param_all):
            order_all = tuple(json.load(open(param_all))['order'])
        else:
            am = auto_arima(
                data_daily['Close'],
                exogenous=exog_all_scaled,
                seasonal=False,
                stepwise=True,
                random_state=seed_value
            )
            order_all = am.order
            json.dump({'order': order_all}, open(param_all, 'w'))

        model_all = ARIMA(data_daily['Close'], order=order_all, exog=exog_all_scaled).fit()
        with open(model_all_path, 'wb') as f:
            pickle.dump(model_all, f)
        print(f"💾 Model keseluruhan disimpan di {model_all_path}")

    preds_all = model_all.predict(
        start=0,
        end=len(data_daily)-1,
        exog=exog_all_scaled
    ).values

    results['Keseluruhan'] = {
        'dates': data_daily.index,
        'actual': data_daily['Close'].values,
        'pred': preds_all
    }
    metrics['Keseluruhan'] = {
        'MSE': mean_squared_error(data_daily['Close'], preds_all),
        'MAE': mean_absolute_error(data_daily['Close'], preds_all),
        'R2': r2_score(data_daily['Close'], preds_all),
        'MAPE': mape(data_daily['Close'].values, preds_all)
    }

    # ==================== CETAK & EKSPOR ====================
    print("\n📊 Evaluasi ARIMA:")
    for p in ['Keseluruhan'] + list(covid_periods.keys()):
        print(f"\n=== {p} ===")
        for k, v in metrics[p].items():
            suf = '%' if k == 'MAPE' else ''
            print(f"{k}: {v:.4f}{suf}")

    export = "export_arima_results"
    os.makedirs(export, exist_ok=True)
    json.dump(metrics, open(os.path.join(export, "all_metrics.json"), 'w'), indent=4)
    for p, res in results.items():
        df_out = pd.DataFrame({
            'Date': res['dates'],
            'Actual': res['actual'],
            'Predicted': res['pred']
        })
        df_out.to_csv(os.path.join(export, f"arima_{p.replace(' ', '_').lower()}.csv"), index=False)

    print("\n✅ Semua hasil ARIMA telah diekspor.")

ℹ️ Memuat scaler dari saved_models_arima\scaler_exog_sebelum_covid.pkl
ℹ️ Memuat model dari saved_models_arima\model_arima_sebelum_covid.pkl


C:\Users\andil\AppData\Local\Temp\ipykernel_15796\1375209289.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  war

ℹ️ Memuat scaler dari saved_models_arima\scaler_exog_selama_covid.pkl
ℹ️ Memuat model dari saved_models_arima\model_arima_selama_covid.pkl


C:\Users\andil\AppData\Local\Temp\ipykernel_15796\1375209289.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  war

ℹ️ Memuat scaler dari saved_models_arima\scaler_exog_setelah_covid.pkl
ℹ️ Memuat model dari saved_models_arima\model_arima_setelah_covid.pkl


C:\Users\andil\AppData\Local\Temp\ipykernel_15796\1375209289.py:62: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  war

ℹ️ Memuat scaler keseluruhan dari saved_models_arima\scaler_exog_keseluruhan.pkl
ℹ️ Memuat model keseluruhan dari saved_models_arima\model_arima_keseluruhan.pkl

📊 Evaluasi ARIMA:

=== Keseluruhan ===
MSE: 1.1003
MAE: 0.6610
R2: 0.9997
MAPE: 0.6212%

=== Sebelum COVID ===
MSE: 0.3951
MAE: 0.4655
R2: 0.9959
MAPE: 0.7112%

=== Selama COVID ===
MSE: 6.3171
MAE: 2.0163
R2: 0.9458
MAPE: 1.3897%

=== Setelah COVID ===
MSE: 14.7812
MAE: 2.5423
R2: 0.9250
MAPE: 1.2193%

✅ Semua hasil ARIMA telah diekspor.
